In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'main'))
sys.path.insert(0, os.path.join('..', 'main', 'utils'))
sys.path.insert(0, os.path.join('..', 'main', 'utils', 'model_training'))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import math

import config
from pipeline_utils import get_exp_paths
from model_utils import (
    build_neighbor_curve_stack,
    reconstruct_curves_cosine,
    _QuerySlice, _AttnScores, _WeightedRecon,
)

In [ ]:
BASE_DATASETS  = "/vol/bitbucket/gk225/POC_DDM_datasets"
EXP_NAME       = "POC_DDM_multi_nc_subtract"
EXP_FOLDER_IDX = 6   # D20260609_E00_C00_F4500KHz_U_norm_temp_ready_08
OUTLIER_FILTER = None
CURVE_TYPE     = "ori_curve"   # alias: ori_curve | ori_curve_avg | ori_curve_wavelet_sym8
N_SAMPLE       = 15
MODEL_NAME     = "cnn_gru_dual_cosine_recon"
K_NEIGHBORS    = 24

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────

def load_experiment_data(base_datasets, exp_name, exp_folder_idx, curve_type, outlier_filter=None):
    """Load curves, coordinates, well_ids and labels for one experiment folder.

    Returns
    -------
    X_AC       : (N, T) float32 — the requested curve type
    coords     : (N, 2) float  — pixel row/col coordinates
    well_ids   : (N,)          — well grouping for neighbour search
    Y_well     : (N,)          — class labels
    exp_path   : Path
    """
    exp_paths = get_exp_paths(os.path.join(base_datasets, exp_name))
    exp_path  = exp_paths[exp_folder_idx]
    print(f"Experiment: {exp_path.name}")

    td = joblib.load(exp_path / config.TRAINING_DATA_PATH)
    dataset_names = td["dataset_name"]
    datasets      = td["dataset"]
    Y_well        = np.array(td["Y_well"])

    # Resolve curve_type alias
    actual_name = config.CURVE_TYPE_ALIASES.get(curve_type, curve_type)
    if actual_name not in dataset_names:
        raise ValueError(f"curve_type '{curve_type}' (→ '{actual_name}') not in dataset. "
                         f"Available: {dataset_names}")
    X_AC = np.array(datasets[dataset_names.index(actual_name)], dtype=np.float32)

    # Coordinates + well grouping
    metadata_df = pd.DataFrame(td["metadata"])
    coords = np.stack([
        metadata_df["pixel_row_idx"].values.astype(float),
        metadata_df["pixel_col_idx"].values.astype(float),
    ], axis=1)
    well_ids = (metadata_df["well_id"].values if "well_id" in metadata_df.columns
                else Y_well.copy())

    # Outlier mask (load the filter's saved mask if requested)
    if outlier_filter is not None:
        filter_path = exp_path / f"{outlier_filter}_outlier" / "outlier_labels.joblib"
        if not filter_path.exists():
            raise FileNotFoundError(f"Outlier mask not found at {filter_path}")
        outlier_labels = joblib.load(filter_path)   # 0 = keep, 1 = outlier
        mask = outlier_labels == 0
        X_AC     = X_AC[mask]
        coords   = coords[mask]
        well_ids = well_ids[mask]
        Y_well   = Y_well[mask]
        print(f"  Outlier filter '{outlier_filter}': kept {mask.sum()} / {len(mask)} samples")

    print(f"  X_AC: {X_AC.shape}, T={X_AC.shape[1]}")
    return X_AC, coords, well_ids, Y_well, exp_path


# ── Reconstruction ────────────────────────────────────────────────────────────

def build_recon_cosine(X_AC, coords, well_ids, k=24):
    """Pure-NumPy cosine-similarity reconstruction. No model needed.

    Returns (neighbor_stack, X_recon, attn_weights=None)
    """
    stack   = build_neighbor_curve_stack(X_AC, coords, well_ids, k=k)  # (N, k+1, T)
    X_recon = reconstruct_curves_cosine(stack)                          # (N, T)
    return stack, X_recon, None


def build_recon_attn(X_AC, coords, well_ids, model_path, k=24):
    """Learned-attention reconstruction from a saved attn_recon keras model.

    Returns (neighbor_stack, X_recon, attn_weights)
      attn_weights : (N, k+1) — softmax weights per neighbour-stack entry
    """
    import tensorflow as tf

    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f"Saved model not found: {model_path}")

    model = tf.keras.models.load_model(
        str(model_path),
        custom_objects={"_QuerySlice": _QuerySlice,
                        "_AttnScores": _AttnScores,
                        "_WeightedRecon": _WeightedRecon},
    )
    print(f"  Loaded model: {model_path.name}  input={model.input_shape}")

    stack = build_neighbor_curve_stack(X_AC, coords, well_ids, k=k)  # (N, k+1, T)

    # Sub-model: full model input → _WeightedRecon output (N, 1, T)
    recon_layer = next(l for l in model.layers if isinstance(l, _WeightedRecon))
    recon_model = tf.keras.Model(model.input, recon_layer.output)
    X_recon = recon_model.predict(stack, verbose=0)[:, 0, :]  # (N, T)

    # Sub-model: full model input → attn_weights (N, 1, k+1)
    attn_layer  = model.get_layer("attn_weights")
    attn_model  = tf.keras.Model(model.input, attn_layer.output)
    attn_weights = attn_model.predict(stack, verbose=0)[:, 0, :]  # (N, k+1)

    return stack, X_recon, attn_weights


def get_reconstruction(X_AC, coords, well_ids, model_name, exp_path,
                       outlier_filter=None, curve_type="ori_curve", k=24):
    """Dispatch to cosine or attn reconstruction based on model_name.

    Returns (neighbor_stack, X_recon, attn_weights or None)
    """
    if "cosine_recon" in model_name:
        return build_recon_cosine(X_AC, coords, well_ids, k=k)

    if "attn_recon" in model_name:
        filter_tag = str(outlier_filter) if outlier_filter is not None else "None"
        model_path = (exp_path / "model_interpretation"
                      / f"{model_name}_{filter_tag}_{curve_type}_model.keras")
        return build_recon_attn(X_AC, coords, well_ids, model_path, k=k)

    raise ValueError(f"'{model_name}' is not a recon model (needs 'cosine_recon' or 'attn_recon')")


# ── Plots ─────────────────────────────────────────────────────────────────────

def plot_recon_grid(neighbor_stack, X_recon, indices, model_name, exp_path,
                   n_cols=5, attn_weights=None):
    """One subplot per sample: neighbour curves (grey) + own curve (blue) + reconstruction (red).

    If attn_weights is provided, neighbour opacity is scaled by attention weight.
    """
    n_rows = math.ceil(len(indices) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 2.8),
                             facecolor="white")
    axes = np.array(axes).flatten()
    T = X_recon.shape[1]
    t = np.arange(T)

    for ax_i, idx in enumerate(indices):
        ax = axes[ax_i]
        # Neighbour curves
        for k_i in range(1, neighbor_stack.shape[1]):
            alpha = (float(attn_weights[idx, k_i]) * 4).clip(0.05, 0.9) if attn_weights is not None else 0.18
            ax.plot(t, neighbor_stack[idx, k_i], color="grey", lw=0.7,
                    linestyle="--", alpha=alpha)
        # Own curve
        ax.plot(t, neighbor_stack[idx, 0], color="steelblue", lw=1.5, label="own")
        # Reconstructed curve
        ax.plot(t, X_recon[idx], color="crimson", lw=1.5, linestyle="-", label="recon")

        ax.set_title(f"#{idx}", fontsize=8)
        ax.tick_params(labelsize=6)
        ax.grid(True, color="grey", alpha=0.2, linestyle=":")
        if ax_i == 0:
            ax.legend(fontsize=7, framealpha=0.8)

    for ax in axes[len(indices):]:
        ax.set_visible(False)

    fig.suptitle(f"{model_name}  |  {exp_path.name}",
                 fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    return fig


def plot_attn_bars(attn_weights, indices, model_name, exp_path, n_cols=5):
    """Bar chart of softmax attention weights per sample (attn_recon only)."""
    if attn_weights is None:
        print("No attention weights available (cosine_recon model).")
        return None

    n_rows = math.ceil(len(indices) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.2),
                             facecolor="white")
    axes = np.array(axes).flatten()
    k_plus_1 = attn_weights.shape[1]

    for ax_i, idx in enumerate(indices):
        ax = axes[ax_i]
        w = attn_weights[idx]               # (k+1,)
        colors = ["steelblue"] + ["grey"] * (k_plus_1 - 1)
        ax.bar(range(k_plus_1), w, color=colors, width=0.8)
        ax.set_title(f"#{idx}", fontsize=8)
        ax.set_xlabel("stack index", fontsize=6)
        ax.tick_params(labelsize=6)
        ax.set_ylim(0, None)

    for ax in axes[len(indices):]:
        ax.set_visible(False)

    fig.suptitle(f"Attention weights  |  {model_name}  |  {exp_path.name}",
                 fontsize=10, fontweight="bold", y=1.01)
    plt.tight_layout()
    return fig

In [ ]:
# Load data
X_AC, coords, well_ids, Y_well, exp_path = load_experiment_data(
    BASE_DATASETS, EXP_NAME, EXP_FOLDER_IDX, CURVE_TYPE, OUTLIER_FILTER
)

# Build reconstruction
neighbor_stack, X_recon, attn_weights = get_reconstruction(
    X_AC, coords, well_ids, MODEL_NAME, exp_path,
    outlier_filter=OUTLIER_FILTER, curve_type=CURVE_TYPE, k=K_NEIGHBORS,
)

print(f"neighbor_stack : {neighbor_stack.shape}")
print(f"X_recon        : {X_recon.shape}")
if attn_weights is not None:
    print(f"attn_weights   : {attn_weights.shape}")

In [ ]:
rng     = np.random.default_rng(42)
indices = rng.choice(len(X_AC), size=min(N_SAMPLE, len(X_AC)), replace=False)
indices.sort()

fig = plot_recon_grid(neighbor_stack, X_recon, indices, MODEL_NAME, exp_path,
                     n_cols=5, attn_weights=attn_weights)
plt.show()

# Attention bar chart (shown only for attn_recon models)
fig2 = plot_attn_bars(attn_weights, indices, MODEL_NAME, exp_path, n_cols=5)
if fig2:
    plt.show()